# Held-out evaluation
Run 01 and 02 (prototype enabled) first. The CNN can be evaluated without training the GRU. Use the same Google Drive folder. Run cells in order. Only evaluate trusted checkpoints produced by this project. Missing CNN checkpoints raise an actionable error. No results are invented. Keep test data untouched during model selection. CPU fallback is supported. Scenario evaluation is optional and explicitly synthetic.


In [ ]:
from pathlib import Path
import os, subprocess, sys
REPO = Path('/content/AI-Crime-Investigation-FYP')
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/ranaalihaider99993-source/AI-Crime-Investigation-FYP.git', str(REPO)], check=True)
os.chdir(REPO)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'], check=True)
import torch
from scripts.common import seed_all
seed_all(42)
print('Device:', 'cuda' if torch.cuda.is_available() else 'cpu')
try:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTPUT = Path('/content/drive/MyDrive/AI-Crime-Investigation-FYP/canonical-four-class-v2')
except ImportError:
    OUTPUT = REPO / 'artifacts'
OUTPUT.mkdir(parents=True, exist_ok=True)
print('Persistent output:', OUTPUT)


In [ ]:
import kagglehub
try:
    DATA = Path(kagglehub.dataset_download('mateohervas/dcsass-dataset'))
except Exception as exc:
    raise RuntimeError('Kaggle download failed. Check network/access; if authentication is required, use Colab secrets or kagglehub.login(), never commit credentials.') from exc
print('Dataset root:', DATA)


In [ ]:
import csv
from scripts.classifier import evaluate
from scripts.scenario import evaluate as evaluate_scenario, infer
required = [OUTPUT / 'classifier/crime_classifier.pth', OUTPUT / 'classifier_splits.csv']
for path in required:
    if not path.is_file():
        raise FileNotFoundError(f'{path}: run the corresponding training notebook first')
# Download cache paths can change after Colab reconnects. Check before evaluation.
with (OUTPUT / 'classifier_splits.csv').open() as f:
    rows = list(csv.DictReader(f))
for row in rows:
    if not Path(row['path']).is_file():
        raise FileNotFoundError('Dataset paths changed. Recreate a reviewed manifest and reproduce the same source split in notebook 02 before evaluation; do not retrain or change test membership.')
classifier_results = evaluate(OUTPUT / 'classifier_splits.csv', required[0], OUTPUT / 'classifier/test_metrics.json')
print(json.dumps(classifier_results, indent=2))
EVALUATE_SYNTHETIC_GRU = False  # Not required for CNN training/evaluation.
if EVALUATE_SYNTHETIC_GRU:
    checkpoint = OUTPUT / 'scenario/scenario_gru.pth'
    if not checkpoint.is_file():
        raise FileNotFoundError('Run notebook 03 only if synthetic GRU evaluation is wanted')
    scenario_results = evaluate_scenario(checkpoint, REPO / 'annotations/synthetic_scenarios.csv')
    (OUTPUT / 'scenario/test_metrics.json').write_text(json.dumps(scenario_results, indent=2))
    print('Synthetic scenario teacher-forced test metrics:', scenario_results)


The classifier report includes accuracy, macro precision/recall/F1, confusion matrix (class order: Burglary, Stealing, Fighting, Vandalism), and per-class support. Zero-division metrics are reported as zero. A balanced prototype does not represent real-world crime prevalence. Neural generation needs human review of uncertainty, evidence consistency, unsupported assertions, and usefulness on independent human-written scenarios. Token accuracy is not scenario correctness. Bounding-box detection is not evaluated because this task does not train a detector.